In [ ]:
from orekit.pyhelpers import download_orekit_data_curdir, setup_orekit_curdir
import orekit


In [2]:
vm = orekit.initVM()
setup_orekit_curdir(from_pip_library=True)
print ('Java version:',vm.java_version)
print ('Orekit version:', orekit.VERSION)

Java version: 1.8.0_472
Orekit version: 13.1


In [3]:
from org.orekit.utils import Constants 
print(Constants.WGS84_EARTH_EQUATORIAL_RADIUS)

6378137.0


all constants are in SI units: seconds, meters, m/s

In [4]:
from org.orekit.bodies import CelestialBodyFactory
from org.orekit.utils import PVCoordinatesProvider
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.frames import FramesFactory

sun = CelestialBodyFactory.getSun()     # Here we get it as an CelestialBody
# sun = PVCoordinatesProvider.cast_(sun)  # But we want the PVCoord interface
sun_frame = sun.getInertiallyOrientedFrame()



earth = CelestialBodyFactory.getEarth()     # Here we get it as an CelestialBody

# Get the position of the sun in the Earth centered coordinate system EME2000
date = AbsoluteDate(2020, 2, 28, 23, 30, 0.0, TimeScalesFactory.getUTC())

# print("Sun in Sun frame:",
#       sun.getPVCoordinates(date, sun_frame).getPosition())

# print("Earth in Sun frame:",
#       earth.getPVCoordinates(date, sun_frame).getPosition())
print(f" Sun location: {sun.getPVCoordinates(date, sun_frame).getPosition()}")
print(f" Earth location: {earth.getPVCoordinates(date, sun_frame).getPosition()}")


 Sun location: {0; 0; 0}
 Earth location: {-120,217,074,065.49298; 84,632,775,489.4906; -18,598,402,224.25442}


# Smoke test for yearly solar system orbit

In [5]:
pos = earth.getPVCoordinates(date, sun_frame).getPosition()
pos.x

-120217074065.49298

In [7]:
import pandas as pd

from org.orekit.bodies import CelestialBodyFactory
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.frames import FramesFactory
from org.orekit.utils import PVCoordinatesProvider

# Bodies
sun = CelestialBodyFactory.getSun()
earth = CelestialBodyFactory.getEarth()
mercury = CelestialBodyFactory.getMercury()
venus = CelestialBodyFactory.getVenus()
mars = CelestialBodyFactory.getMars()
jupiter = CelestialBodyFactory.getJupiter()
saturn = CelestialBodyFactory.getSaturn()
uranus = CelestialBodyFactory.getUranus()
neptune = CelestialBodyFactory.getNeptune()  # fixed this

celestial_bodies = {
    "Sun": sun,
    "Mercury": mercury,
    "Venus": venus,
    "Earth": earth,
    "Mars": mars,
    "Jupiter": jupiter,
    "Saturn": saturn,
    "Uranus": uranus,
    "Neptune": neptune,
}

# Frame and time scale
frame = sun.getInertiallyOrientedFrame()  # Sun-centered inertial frame
utc = TimeScalesFactory.getUTC()

# Time settings
start_date = AbsoluteDate(2026, 1, 1, 0, 0, 0.0, utc)
num_days = 365
step_seconds = 24 * 60 * 60  # one day

rows = []

for i in range(num_days + 1):
    date = start_date.shiftedBy(float(i * step_seconds))
    for name, body in celestial_bodies.items():
        pv_provider = PVCoordinatesProvider.cast_(body)
        position = pv_provider.getPVCoordinates(date, frame).getPosition()

        rows.append({
            "date": date.toString(),
            "body": name,
            "x_m": position.getX(),
            "y_m": position.getY(),
            "z_m": position.getZ(),
        })

df = pd.DataFrame(rows)

df.to_csv("planet_positions.csv", index=False)

df.head()

,date,body,x_m,y_m,z_m
0,2026-01-01T00:00:00.000Z,Sun,0.000000e+00,0.000000e+00,0.000000e+00
1,2026-01-01T00:00:00.000Z,Mercury,-4.630155e+10,-5.126120e+10,-4.072554e+09
2,2026-01-01T00:00:00.000Z,Venus,-1.449829e+10,-1.078078e+11,2.745674e+09
3,2026-01-01T00:00:00.000Z,Earth,1.185514e+10,1.464232e+11,-7.692577e+09
4,2026-01-01T00:00:00.000Z,Mars,-3.323708e+09,-2.135851e+11,7.121433e+09


# Experiment 1

The paper has a few experiments of placing a spacecraft in different orbits and computing the amount of time the spacecraft ends up in umbra and penumbra, the experiment below serves as a way to recreate the results and validate our implementation

In [ ]:
import math

from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.frames import FramesFactory
from org.orekit.orbits import KeplerianOrbit, PositionAngleType
from org.orekit.propagation.analytical import KeplerianPropagator
from org.orekit.utils import Constants

utc = TimeScalesFactory.getUTC()
frame = FramesFactory.getEME2000()

date0 = AbsoluteDate(1994, 3, 20, 0, 0, 0.0, utc)

mu = Constants.EIGEN5C_EARTH_MU

a = 44859.14e3
e = 0.8408
i = math.radians(4.47)
pa = math.radians(90.0)
raan = math.radians(-90.0)
anomaly = math.radians(0.0)

orbit = KeplerianOrbit(
    a,
    e,
    i,
    pa,
    raan,
    anomaly,
    PositionAngleType.TRUE,
    frame,
    date0,
    mu
)

propagator = KeplerianPropagator(orbit)